In [1]:
%pip uninstall --yes 'keras' 'matplotlib' 'scikit-learn' 'tensorflow'

Found existing installation: keras 3.10.0
Uninstalling keras-3.10.0:
  Successfully uninstalled keras-3.10.0
Found existing installation: matplotlib 3.10.0
Uninstalling matplotlib-3.10.0:
  Successfully uninstalled matplotlib-3.10.0
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
Found existing installation: tensorflow 2.19.0
Uninstalling tensorflow-2.19.0:
  Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
import os
import sys
import subprocess

In [4]:
def set_env(input_archive, temp_dir):

    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True)
        
        subprocess.run(['tar', '-xzf', input_archive, '-C', temp_dir], check=True)
    
    subprocess.run([
        sys.executable, 
        '-m', 
        'pip', 
        'install', 
        '--no-index', 
        '--find-links', 
        f'{temp_dir}/wheels', 
        'unsloth', 
        'trl', 
        'vllm', 
        'openai_harmony'
    ], check=True)

In [5]:
set_env(
    input_archive='/kaggle/input/aimo-3-utils/wheels.tar.gz', 
    temp_dir='/kaggle/tmp/setup'
)

Looking in links: /kaggle/tmp/setup/wheels
Processing /kaggle/tmp/setup/wheels/unsloth-2025.12.9-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/trl-0.24.0-py3-none-any.whl
Processing /kaggle/tmp/setup/wheels/vllm-0.11.2-cp38-abi3-manylinux1_x86_64.whl
Processing /kaggle/tmp/setup/wheels/openai_harmony-0.0.8-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Processing /kaggle/tmp/setup/wheels/unsloth_zoo-2025.12.7-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/tyro-1.0.3-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/xformers-0.0.33.post1-cp39-abi3-manylinux_2_28_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/bitsandbytes-0.49.0-py3-none-manylinux_2_24_x86_64.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/datasets-4.3.0-py3-none-any.whl (from unsloth)
Processing /kaggle/tmp/setup/wheels/prometheus_fastapi_instrumentator-7.1.0-py3-none-any.whl (from vllm)
Processing /kaggle/tmp/setup/wheels/lm_format_enforcer-0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
kauldron 1.3.0 requires scikit-learn, which is not installed.
kauldron 1.3.0 requires tensorflow, which is not installed.
ydata-profiling 4.18.0 requires matplotlib<=3.10,>=3.5, which is not installed.
pyldavis 3.4.1 requires scikit-learn>=1.0.0, which is not installed.
stable-baselines3 2.1.0 requires matplotlib, which is not installed.
sentence-transformers 5.1.1 requires scikit-learn, which is not installed.
librosa 0.11.0 requires scikit-learn>=1.1.0, which is not installed.
cuml-cu12 25.6.0 requires scikit-learn>=1.5, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.26.0 requires matplotlib>=3.7.1, which is not installed.
arviz 0.22.0 requires matplotlib>=3.8, which is not installed.
pynndescent 0.5.13 requires scikit-learn>=0.

In [6]:
subprocess.run(['ls', '/kaggle/tmp/setup/tiktoken_encodings'])

cl100k_base.tiktoken
o200k_base.tiktoken


CompletedProcess(args=['ls', '/kaggle/tmp/setup/tiktoken_encodings'], returncode=0)

In [7]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = '/kaggle/tmp/setup/tiktoken_encodings'

In [8]:
import gc
import re
import math
import time
import queue
import threading
import contextlib
from typing import Any, Optional
from jupyter_client import KernelManager
from collections import Counter, defaultdict
from concurrent.futures import as_completed, ThreadPoolExecutor

import pandas as pd
import polars as pl

from openai import OpenAI

from openai_harmony import (
    HarmonyEncodingName, 
    load_harmony_encoding, 
    SystemContent, 
    ReasoningEffort, 
    ToolNamespaceConfig, 
    Author, 
    Message, 
    Role, 
    TextContent, 
    Conversation
)

from transformers import set_seed



In [9]:
class CFG:
    
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'
        
        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'
        
        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n\n'
        
        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n\n'
        
        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'
        
        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )
    
    tool_prompt = (
        'Use this tool to execute Python code for:\n'
        '- Complex calculations that would be error-prone by hand\n'
        '- Numerical verification of analytical results\n'
        '- Generating examples or testing conjectures\n'
        '- Visualizing problem structure when helpful\n'
        '- Brute-force verification for small cases\n\n'
        
        'The environment is a stateful Jupyter notebook. Code persists between executions.\n'
        'Always use print() to display results. Write clear, well-commented code.\n\n'
        
        'Remember: Code should support your mathematical reasoning, not replace it. '
        'Explain what you\'re computing and why before running code.'
    )
    
    preference_prompt = (
        'You have access to `math`, `numpy`, and `sympy` for:\n\n'
        
        '# Symbolic Computation (sympy):\n'
        '- Algebraic manipulation and simplification\n'
        '- Solving equations and systems of equations\n'
        '- Symbolic differentiation and integration\n'
        '- Number theory functions (primes, divisors, modular arithmetic)\n'
        '- Polynomial operations and factorization\n'
        '- Working with mathematical expressions symbolically\n\n'
        
        '# Numerical Computation (numpy):\n'
        '- Array operations and linear algebra\n'
        '- Efficient numerical calculations for large datasets\n'
        '- Matrix operations and eigenvalue problems\n'
        '- Statistical computations\n\n'
        
        '# Mathematical Functions (math):\n'
        '- Standard mathematical functions (trig, log, exp)\n'
        '- Constants like pi and e\n'
        '- Basic operations for single values\n\n'
        
        'Best Practices:\n'
        '- Use sympy for exact symbolic answers when possible\n'
        '- Use numpy for numerical verification and large-scale computation\n'
        '- Combine symbolic and numerical approaches: derive symbolically, verify numerically\n'
        '- Document your computational strategy clearly\n'
        '- Validate computational results against known cases or theoretical bounds'
    )
    
    served_model_name = 'gpt-oss'
    model_path = '/kaggle/input/gpt-oss-120b/transformers/default/1'
    
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'

    high_problem_timeout = 900
    base_problem_timeout = 300

    notebook_limit = 17400
    server_timeout = 180

    session_timeout = 960
    jupyter_timeout = 6
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 65536
    buffer_tokens = 512
    search_tokens = 32
    top_logprobs = 5
    batch_size = 256
    early_stop = 4
    attempts = 8
    workers = 16
    turns = 128
    seed = 42

    gpu_memory_utilization = 0.96
    temperature = 1.0
    min_p = 0.02

In [10]:
set_seed(CFG.seed)

In [11]:
class AIMO3Template:

    def __init__(self):

        pass

    def get_system_content(self, system_prompt: str, tool_config: ToolNamespaceConfig) -> SystemContent:

        return (
            SystemContent.new()
            .with_model_identity(system_prompt)
            .with_reasoning_effort(reasoning_effort=ReasoningEffort.HIGH)
            .with_tools(tool_config)
        )

    def apply_chat_template(
        self, 
        system_prompt: str, 
        user_prompt: str, 
        tool_config: ToolNamespaceConfig
    ) -> list[Message]:

        system_content = self.get_system_content(system_prompt, tool_config)        
        system_message = Message.from_role_and_content(Role.SYSTEM, system_content)

        user_message = Message.from_role_and_content(Role.USER, user_prompt)

        return [system_message, user_message]

In [12]:
class AIMO3Sandbox:

    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None
        
        ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()
        self._km.shell_port = ports[0]
        self._km.iopub_port = ports[1]
        self._km.stdin_port = ports[2]
        self._km.hb_port = ports[3]
        self._km.control_port = ports[4]

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout
        
        msg_id = client.execute(
            code, 
            store_history=True, 
            allow_stdin=False, 
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []
        
        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        with contextlib.suppress(Exception):
            if self._client:
                self._client.stop_channels()

        if self._owns_kernel and self._km is not None:
            with contextlib.suppress(Exception):
                self._km.shutdown_kernel(now=True)

            with contextlib.suppress(Exception):
                self._km.cleanup_resources()

    def reset(self):
        
        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

In [13]:
class AIMO3Tool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox
        
        self._owns_session = sandbox is None
        
        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = AIMO3Sandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if 'print' in last_line or 'import' in last_line:
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    @property
    def tool_config(self) -> ToolNamespaceConfig:

        return ToolNamespaceConfig(
            name='python', 
            description=self.instruction, 
            tools=[]
        )

    def _make_response(self, output: str, channel: str | None = None) -> Message:

        content = TextContent(text=output)
        author = Author(role=Role.TOOL, name='python')
        message = Message(author=author, content=[content]).with_recipient('assistant')

        if channel:
            message = message.with_channel(channel)

        return message

    def process_sync_plus(self, message: Message) -> list[Message]:

        self._ensure_session()
        raw_script = message.content[0].text
        final_script = self._ensure_last_print(raw_script)

        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] {exc}'

        return [self._make_response(output, channel=message.channel)]

In [14]:
from dataclasses import dataclass, field

EMBEDDED_PROMPTS = {
    "baseline": 'You are solving a mathematical olympiad-style problem.\n\nWork carefully and keep the solution mathematically grounded.\nPrefer exact reasoning over guesswork.\nIf a direct attack looks messy, simplify the structure and solve the core subproblem first.\nTrack any modulus or answer-range instruction explicitly.\nThe final answer must be a single integer.\nEnd with a clearly marked final answer.',
    "tool_guided": 'Use the Python tool only when it materially helps.\n\nGood uses:\n- checking small cases\n- testing a conjectured pattern\n- evaluating a closed form\n- verifying arithmetic, modular reduction, or brute-force bounds on a reduced search space\n\nAvoid using Python as a substitute for all reasoning.\nBefore using the tool, say what you are checking.\nAfter using the tool, explain how the result supports the mathematical argument.\nIf the problem involves huge exponents or recurrences, prefer exploring smaller cases first and then generalizing.',
    "reflexion_self_refute": 'You are solving a mathematical olympiad-style problem with an explicit self-refutation pass.\n\nProduce a candidate solution.\nThen attack your own solution:\n- look for counterexamples\n- test edge cases\n- question any unjustified symmetry or divisibility claim\n- check whether a different interpretation of the problem changes the result\n\nIf the candidate survives the attack, keep it.\nIf it fails, repair the reasoning and continue.\nTrack modulus and output-range instructions explicitly.\nThe final answer must be a single integer.\nEnd with a clearly marked final answer.',
}

PROMPT_FAMILY_SPECS = {
    "baseline": {"variant_name": "baseline", "include_tool_guidance": False},
    "tool_guided": {"variant_name": "baseline", "include_tool_guidance": True},
    "self_refute": {"variant_name": "reflexion_self_refute", "include_tool_guidance": False},
}

PROMPT_FAMILY_ALIASES = {
    "baseline_tir": "baseline",
    "reflexion_rigor": "self_refute",
    "large_number": "baseline",
}


def load_embedded_prompt_template(name: str) -> str:
    if name not in EMBEDDED_PROMPTS:
        raise ValueError(f"unknown embedded prompt template: {name}")
    return EMBEDDED_PROMPTS[name]


def compose_prompt(*, system_prompt: str, tool_guidance: str | None = None, task_hint: str | None = None) -> str:
    sections = [system_prompt.strip()]
    if tool_guidance:
        sections.append(tool_guidance.strip())
    if task_hint:
        sections.append(f"Task hint:\n{task_hint.strip()}")
    return "\n\n".join(section for section in sections if section)


@dataclass(frozen=True, slots=True)
class PromptRoute:
    family_name: str
    variant_name: str
    include_tool_guidance: bool
    task_hint: str | None
    full_prompt: str


def normalize_prompt_family(family_name: str) -> str:
    return PROMPT_FAMILY_ALIASES.get(family_name, family_name)


def build_prompt_routes(family_sequence: tuple[str, ...] | list[str], *, task_hint: str | None = None) -> list[PromptRoute]:
    routes: list[PromptRoute] = []
    for family_name in family_sequence:
        normalized_family = normalize_prompt_family(family_name)
        spec = PROMPT_FAMILY_SPECS.get(normalized_family)
        if spec is None:
            raise ValueError(f"unknown prompt family: {family_name}")
        system_prompt = load_embedded_prompt_template(spec["variant_name"])
        tool_text = load_embedded_prompt_template("tool_guided") if spec["include_tool_guidance"] else None
        full_prompt = compose_prompt(system_prompt=system_prompt, tool_guidance=tool_text, task_hint=task_hint)
        routes.append(
            PromptRoute(
                family_name=normalized_family,
                variant_name=spec["variant_name"],
                include_tool_guidance=spec["include_tool_guidance"],
                task_hint=task_hint,
                full_prompt=full_prompt,
            )
        )
    return routes


def build_prompt_routes_for_policy(policy: SolvePolicy, *, task_hint: str | None = None) -> list[PromptRoute]:
    return build_prompt_routes(policy.prompt_families, task_hint=task_hint)


def build_retry_prompt_route(policy: SolvePolicy, retry_decision: RetryDecision, *, task_hint: str | None = None) -> PromptRoute | None:
    if not retry_decision.should_retry or retry_decision.prompt_family is None:
        return None
    return build_prompt_routes([retry_decision.prompt_family], task_hint=task_hint)[0]

from dataclasses import dataclass, field


@dataclass(slots=True)
class ParseResult:
    raw_answer: int | None
    normalized_answer: int | None
    method: str
    success: bool
    confidence: float
    notes: list[str] = field(default_factory=list)


@dataclass(slots=True)
class ModulusResult:
    modulus: int | None
    source: str
    matched_text: str | None
    success: bool
    notes: list[str] = field(default_factory=list)

import re



DEFAULT_FINAL_ANSWER_MODULUS = 100_000
POWER_PATTERN = re.compile(r"(?P<base>\d+)\s*\^\s*(?P<exponent>\d+)")
PATTERN_SPECS = (
    ("modulo", re.compile(r"\bmodulo\s+(?P<value>\d+(?:\s*\^\s*\d+)?)", re.IGNORECASE)),
    ("mod", re.compile(r"\bmod\s+(?P<value>\d+(?:\s*\^\s*\d+)?)", re.IGNORECASE)),
    (
        "remainder_when_divided_by",
        re.compile(r"\bremainder\s+when\s+divided\s+by\s+(?P<value>\d+)", re.IGNORECASE),
    ),
    (
        "remainder_when_expression_is_divided_by",
        re.compile(
            r"\bremainder\s+when\b.*?\bis\s+divided\s+by\s+(?P<value>\d+)",
            re.IGNORECASE,
        ),
    ),
    (
        "final_answer_range",
        re.compile(
            r"\bfinal\s+answer\s+should\s+be\s+in\s*\[\s*0\s*,\s*99999\s*\]",
            re.IGNORECASE,
        ),
    ),
)


def detect_modulus(problem_text: str) -> ModulusResult:
    if not isinstance(problem_text, str):
        raise TypeError("problem_text must be a string")

    notes: list[str] = []

    for source, pattern in PATTERN_SPECS:
        match = pattern.search(problem_text)
        if match is None:
            continue

        matched_text = match.group(0)
        if source == "final_answer_range":
            notes.append("used fixed normalization modulus for required final answer range")
            return ModulusResult(
                modulus=DEFAULT_FINAL_ANSWER_MODULUS,
                source=source,
                matched_text=matched_text,
                success=True,
                notes=notes,
            )

        raw_value = match.group("value")
        modulus = _parse_modulus_value(raw_value)
        if modulus is None:
            notes.append(f"matched modulus instruction but could not parse value: {raw_value}")
            return ModulusResult(
                modulus=None,
                source=source,
                matched_text=matched_text,
                success=False,
                notes=notes,
            )

        return ModulusResult(
            modulus=modulus,
            source=source,
            matched_text=matched_text,
            success=True,
            notes=notes,
        )

    notes.append("no modulus instruction detected")
    return ModulusResult(
        modulus=None,
        source="none",
        matched_text=None,
        success=False,
        notes=notes,
    )


def _parse_modulus_value(raw_value: str) -> int | None:
    stripped = raw_value.strip()
    if stripped.isdigit():
        return int(stripped)

    power_match = POWER_PATTERN.fullmatch(stripped)
    if power_match is None:
        return None

    base = int(power_match.group("base"))
    exponent = int(power_match.group("exponent"))
    return base**exponent

import re



DEFAULT_NORMALIZATION_MODULUS = 100_000
BOXED_PREFIX = r"\boxed{"
INTEGER_PATTERN = re.compile(r"[-+]?\d+")


def normalize_answer(answer: int, modulus: int = DEFAULT_NORMALIZATION_MODULUS) -> int:
    if modulus <= 0:
        raise ValueError("modulus must be positive")
    return answer % modulus


def parse_model_output(
    raw_output: str,
    *,
    normalization_modulus: int = DEFAULT_NORMALIZATION_MODULUS,
) -> ParseResult:
    if not isinstance(raw_output, str):
        raise TypeError("raw_output must be a string")

    notes: list[str] = []

    boxed_content = _extract_last_boxed_content(raw_output)
    if boxed_content is not None:
        boxed_value = _parse_integer_text(boxed_content)
        if boxed_value is not None:
            return ParseResult(
                raw_answer=boxed_value,
                normalized_answer=normalize_answer(boxed_value, normalization_modulus),
                method="boxed",
                success=True,
                confidence=0.95,
                notes=notes,
            )
        notes.append("found boxed answer but its contents were not a valid integer")

    integers = INTEGER_PATTERN.findall(raw_output)
    if integers:
        raw_answer = int(integers[-1])
        if boxed_content is None:
            notes.append("boxed answer not found; used last integer fallback")
        else:
            notes.append("used last integer fallback after invalid boxed answer")
        return ParseResult(
            raw_answer=raw_answer,
            normalized_answer=normalize_answer(raw_answer, normalization_modulus),
            method="last_integer",
            success=True,
            confidence=0.6,
            notes=notes,
        )

    if boxed_content is None:
        notes.append("no boxed answer found")
    notes.append("no integer answer found in model output")
    return ParseResult(
        raw_answer=None,
        normalized_answer=None,
        method="none",
        success=False,
        confidence=0.0,
        notes=notes,
    )


def _parse_integer_text(value: str) -> int | None:
    candidate = value.strip()
    if not candidate:
        return None
    if not re.fullmatch(r"[-+]?\d+", candidate):
        return None
    return int(candidate)


def _extract_last_boxed_content(raw_output: str) -> str | None:
    start = 0
    last_content: str | None = None

    while True:
        prefix_index = raw_output.find(BOXED_PREFIX, start)
        if prefix_index == -1:
            return last_content

        content_start = prefix_index + len(BOXED_PREFIX)
        depth = 1
        index = content_start

        while index < len(raw_output) and depth > 0:
            char = raw_output[index]
            if char == "{":
                depth += 1
            elif char == "}":
                depth -= 1
            index += 1

        if depth == 0:
            last_content = raw_output[content_start : index - 1]
            start = index
            continue

        return last_content

from dataclasses import dataclass, field
import re



@dataclass(slots=True)
class VerifyFixResult:
    status: str
    retry_recommended: bool
    notes: list[str] = field(default_factory=list)
    retry_reasons: list[str] = field(default_factory=list)
    soft_flags: list[str] = field(default_factory=list)
    suspicion_score: float = 0.0


def verify_candidate(
    problem_text: str,
    candidate_answer: int | None,
    parser_result: ParseResult,
    modulus_result: ModulusResult,
    raw_reasoning_text: str | None = None,
) -> VerifyFixResult:
    if not isinstance(problem_text, str):
        raise TypeError("problem_text must be a string")

    retry_reasons: list[str] = []
    soft_flags: list[str] = []

    if not isinstance(candidate_answer, int) or isinstance(candidate_answer, bool):
        retry_reasons.append("candidate answer is not an integer")

    if not parser_result.success:
        retry_reasons.append("parser did not produce a successful answer")

    if parser_result.method != "boxed":
        soft_flags.append("parse ambiguity flagged: answer did not come from boxed extraction")

    if any("fallback" in note or "invalid boxed" in note for note in parser_result.notes):
        soft_flags.append("parse ambiguity flagged: parser notes indicate fallback behavior")

    if raw_reasoning_text and raw_reasoning_text.count(r"\boxed{") > 1:
        soft_flags.append("parse ambiguity flagged: multiple boxed answers appeared in reasoning")

    if isinstance(candidate_answer, int) and not isinstance(candidate_answer, bool):
        if modulus_result.success and modulus_result.modulus is not None:
            expected_min = 0
            expected_max = modulus_result.modulus - 1
            if not expected_min <= candidate_answer <= expected_max:
                retry_reasons.append(
                    f"answer is outside expected range [0, {expected_max}]"
                )

            normalized_candidate = candidate_answer % modulus_result.modulus
            if candidate_answer != normalized_candidate:
                retry_reasons.append("candidate answer is not modulus-normalized")

            if (
                parser_result.normalized_answer is not None
                and parser_result.normalized_answer != normalized_candidate
            ):
                retry_reasons.append("parser normalization is inconsistent with modulus result")

        if _looks_suspiciously_tiny(problem_text, candidate_answer):
            soft_flags.append("suspiciously tiny answer flagged for review")

    suspicion_score = (1.5 * len(retry_reasons)) + (0.5 * len(soft_flags))
    notes = [*retry_reasons, *soft_flags]

    if retry_reasons:
        return VerifyFixResult(
            status="retry",
            retry_recommended=True,
            notes=notes,
            retry_reasons=retry_reasons,
            soft_flags=soft_flags,
            suspicion_score=suspicion_score,
        )

    if soft_flags:
        return VerifyFixResult(
            status="flagged",
            retry_recommended=any("parse ambiguity" in flag for flag in soft_flags),
            notes=notes,
            retry_reasons=[],
            soft_flags=soft_flags,
            suspicion_score=suspicion_score,
        )

    return VerifyFixResult(
        status="ok",
        retry_recommended=False,
        notes=["verification checks passed"],
        retry_reasons=[],
        soft_flags=[],
        suspicion_score=0.0,
    )


def _looks_suspiciously_tiny(problem_text: str, candidate_answer: int) -> bool:
    if abs(candidate_answer) > 2:
        return False
    return bool(re.search(r"10\s*\^|\b\d{4,}\b", problem_text))

from dataclasses import dataclass, field
import re
from typing import Any



FINALITY_PATTERNS = (
    re.compile(r"\\boxed\s*\{", re.IGNORECASE),
    re.compile(r"\bfinal answer\b", re.IGNORECASE),
    re.compile(r"\bthe answer is\b", re.IGNORECASE),
)

PARSE_TIER_RANKS = {
    "none": 0,
    "tier3": 1,
    "tier2": 2,
    "tier1": 3,
}


@dataclass(slots=True)
class AttemptRecord:
    attempt_index: int
    prompt_family: str
    raw_problem_text: str
    prompted_problem_text: str
    raw_output_text: str
    parsed_answer: int | None
    parser_result: ParseResult
    modulus_result: ModulusResult
    verify_result: VerifyFixResult
    parse_tier: str
    finality_score: float
    tool_used: bool = False
    python_calls: int = 0
    python_errors: int = 0
    response_length: int = 0
    entropy: float = float("inf")
    finish_reason: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)

    @property
    def valid_candidate(self) -> bool:
        return self.parsed_answer is not None and self.verify_result.status != "retry"

    @property
    def verify_status(self) -> str:
        return self.verify_result.status

    @property
    def retry_reasons(self) -> list[str]:
        return self.verify_result.retry_reasons

    @property
    def suspicious_flags(self) -> list[str]:
        return self.verify_result.soft_flags

    @property
    def tool_consistency(self) -> bool:
        return self.tool_used and self.python_errors == 0

    @property
    def extraction_failed(self) -> bool:
        return self.parsed_answer is None

    @property
    def parse_tier_rank(self) -> int:
        return PARSE_TIER_RANKS.get(self.parse_tier, 0)

    @property
    def policy_retry_reasons(self) -> tuple[str, ...]:
        reasons: list[str] = []

        if self.parse_tier in {"none", "tier3"} or self.extraction_failed:
            reasons.append("weak_parse")

        if any("multiple boxed answers" in flag.lower() for flag in self.suspicious_flags):
            reasons.append("multiple_boxed")

        if any("suspiciously tiny" in flag.lower() for flag in self.suspicious_flags):
            reasons.append("suspicious_tiny_answer")

        if self.tool_used and not self.tool_consistency:
            reasons.append("tool_mismatch")

        if self.verify_status == "retry":
            reasons.append("inconsistent_reasoning")

        if not reasons and self.verify_status == "flagged":
            reasons.append("inconsistent_reasoning")

        return tuple(dict.fromkeys(reasons))


def build_attempt_record(
    *,
    attempt_index: int,
    prompt_family: str,
    raw_problem_text: str,
    prompted_problem_text: str,
    raw_output_text: str,
    tool_used: bool = False,
    python_calls: int = 0,
    python_errors: int = 0,
    response_length: int = 0,
    entropy: float = float("inf"),
    finish_reason: str | None = None,
    metadata: dict[str, Any] | None = None,
) -> AttemptRecord:
    modulus_result = detect_modulus(raw_problem_text)
    parser_result = parse_model_output(raw_output_text)
    parsed_answer = parser_result.normalized_answer if parser_result.success else None
    verify_result = verify_candidate(
        problem_text=raw_problem_text,
        candidate_answer=parsed_answer,
        parser_result=parser_result,
        modulus_result=modulus_result,
        raw_reasoning_text=raw_output_text,
    )
    finality_score = compute_finality_score(raw_output_text)
    parse_tier = classify_parse_tier(parser_result, finality_score=finality_score)

    return AttemptRecord(
        attempt_index=attempt_index,
        prompt_family=prompt_family,
        raw_problem_text=raw_problem_text,
        prompted_problem_text=prompted_problem_text,
        raw_output_text=raw_output_text,
        parsed_answer=parsed_answer,
        parser_result=parser_result,
        modulus_result=modulus_result,
        verify_result=verify_result,
        parse_tier=parse_tier,
        finality_score=finality_score,
        tool_used=tool_used,
        python_calls=python_calls,
        python_errors=python_errors,
        response_length=response_length,
        entropy=entropy,
        finish_reason=finish_reason,
        metadata=dict(metadata or {}),
    )


def classify_parse_tier(parser_result: ParseResult, *, finality_score: float) -> str:
    if not parser_result.success:
        return "none"

    if parser_result.method == "boxed" and finality_score >= 1.0:
        return "tier1"

    if parser_result.method == "boxed":
        return "tier2"

    if parser_result.method == "last_integer" and finality_score >= 1.0:
        return "tier2"

    return "tier3"


def compute_finality_score(raw_output_text: str) -> float:
    score = 0.0
    for pattern in FINALITY_PATTERNS:
        if pattern.search(raw_output_text):
            score += 0.6
    return min(score, 1.8)

from collections import Counter
from dataclasses import dataclass, field



PARSE_TIER_SCORES = {
    "tier1": 3.0,
    "tier2": 2.0,
    "tier3": 0.5,
    "none": -2.0,
}

VERIFY_STATUS_SCORES = {
    "ok": 2.5,
    "flagged": 1.0,
    "retry": -3.0,
}


@dataclass(slots=True)
class SelectionBreakdown:
    attempt_index: int
    answer: int | None
    total_score: float
    consensus_score: float
    parse_score: float
    verify_score: float
    finality_score: float
    tool_score: float
    entropy_score: float
    suspicion_penalty: float


@dataclass(slots=True)
class SelectionResult:
    selected_answer: int | None
    selected_attempt: AttemptRecord | None
    scored_attempts: list[SelectionBreakdown] = field(default_factory=list)


def select_best_attempt(attempts: list[AttemptRecord]) -> SelectionResult:
    # Current Sprint 1/2 selector corresponds to the "verifier_weighted"
    # selection mode from the policy layer.
    if not attempts:
        return SelectionResult(selected_answer=None, selected_attempt=None, scored_attempts=[])

    valid_answers = [attempt.parsed_answer for attempt in attempts if attempt.parsed_answer is not None]
    answer_counts = Counter(valid_answers)
    max_count = max(answer_counts.values(), default=0)

    scored: list[tuple[AttemptRecord, SelectionBreakdown]] = []
    for attempt in attempts:
        breakdown = _score_attempt(attempt, answer_counts=answer_counts, max_count=max_count)
        scored.append((attempt, breakdown))

    scored.sort(
        key=lambda item: (
            item[1].total_score,
            item[0].finality_score,
            -item[0].entropy,
            -(item[0].tool_used and item[0].python_errors == 0),
        ),
        reverse=True,
    )

    best_attempt = scored[0][0]
    selected_answer = best_attempt.parsed_answer if best_attempt.valid_candidate else None
    if selected_answer is None and answer_counts:
        selected_answer = answer_counts.most_common(1)[0][0]

    return SelectionResult(
        selected_answer=selected_answer,
        selected_attempt=best_attempt,
        scored_attempts=[breakdown for _, breakdown in scored],
    )


def _score_attempt(
    attempt: AttemptRecord,
    *,
    answer_counts: Counter[int],
    max_count: int,
) -> SelectionBreakdown:
    # Sprint 1 stays heuristic and inspectable on purpose.
    # Later stability and retry-aware signals can be added here without
    # changing the caller contract or the breakdown shape.
    consensus_count = answer_counts.get(attempt.parsed_answer, 0) if attempt.parsed_answer is not None else 0
    consensus_score = (consensus_count / max_count) * 3.0 if max_count else 0.0
    parse_score = PARSE_TIER_SCORES.get(attempt.parse_tier, -2.0)
    verify_score = VERIFY_STATUS_SCORES.get(attempt.verify_result.status, -1.0)
    finality_score = min(attempt.finality_score, 1.5)
    tool_score = 0.75 if attempt.tool_consistency else 0.0
    entropy_score = _entropy_score(attempt.entropy)
    suspicion_penalty = min(attempt.verify_result.suspicion_score, 3.0)

    total_score = (
        consensus_score
        + parse_score
        + verify_score
        + finality_score
        + tool_score
        + entropy_score
        - suspicion_penalty
    )

    return SelectionBreakdown(
        attempt_index=attempt.attempt_index,
        answer=attempt.parsed_answer,
        total_score=total_score,
        consensus_score=consensus_score,
        parse_score=parse_score,
        verify_score=verify_score,
        finality_score=finality_score,
        tool_score=tool_score,
        entropy_score=entropy_score,
        suspicion_penalty=suspicion_penalty,
    )


def _entropy_score(entropy: float) -> float:
    if entropy == float("inf"):
        return 0.0
    if entropy <= 0.75:
        return 1.0
    if entropy <= 1.5:
        return 0.6
    if entropy <= 3.0:
        return 0.2
    return 0.0

from collections import Counter
from dataclasses import dataclass
import re



SELECTION_MODE_VERIFIER_WEIGHTED = "verifier_weighted"
TOOL_USE_OPTIONAL = "optional"
TOOL_USE_ENCOURAGED = "encouraged"
TOOL_USE_REQUIRED = "required"


@dataclass(frozen=True, slots=True)
class AttemptPolicy:
    attempt_count: int
    prompt_family_distribution: tuple[str, ...]
    tool_use_mode: str = TOOL_USE_ENCOURAGED

    def __post_init__(self) -> None:
        if self.attempt_count <= 0:
            raise ValueError("attempt_count must be positive")
        if len(self.prompt_family_distribution) != self.attempt_count:
            raise ValueError("prompt_family_distribution length must match attempt_count")


@dataclass(frozen=True, slots=True)
class EarlyStopPolicy:
    min_matching_answers: int
    min_parse_tier: str
    allowed_verify_statuses: tuple[str, ...]
    require_tool_support: bool = False


@dataclass(frozen=True, slots=True)
class RetryPolicy:
    retry_budget: int
    allowed_reasons: tuple[str, ...]
    reason_prompt_family_map: tuple[tuple[str, str], ...]


@dataclass(frozen=True, slots=True)
class SelectionPolicy:
    mode: str = SELECTION_MODE_VERIFIER_WEIGHTED


@dataclass(frozen=True, slots=True)
class SolvePolicy:
    name: str
    attempt_policy: AttemptPolicy
    early_stop_policy: EarlyStopPolicy
    retry_policy: RetryPolicy
    selection_policy: SelectionPolicy

    @property
    def attempt_count(self) -> int:
        return self.attempt_policy.attempt_count

    @property
    def prompt_families(self) -> tuple[str, ...]:
        return self.attempt_policy.prompt_family_distribution

    @property
    def early_stop_consensus(self) -> int:
        return self.early_stop_policy.min_matching_answers

    @property
    def retry_budget(self) -> int:
        return self.retry_policy.retry_budget

    @property
    def require_verify_for_early_stop(self) -> bool:
        return bool(self.early_stop_policy.allowed_verify_statuses)


@dataclass(frozen=True, slots=True)
class PolicySet:
    easy: SolvePolicy
    medium: SolvePolicy
    hard: SolvePolicy

    def for_problem(self, problem_text: str) -> SolvePolicy:
        difficulty = classify_problem_difficulty(problem_text)
        if difficulty == "easy":
            return self.easy
        if difficulty == "hard":
            return self.hard
        return self.medium


@dataclass(frozen=True, slots=True)
class EarlyStopDecision:
    should_stop: bool
    answer: int | None
    matching_attempts: tuple[int, ...]
    reason: str | None


@dataclass(frozen=True, slots=True)
class RetryDecision:
    should_retry: bool
    reason: str | None
    prompt_family: str | None
    remaining_budget: int


GEOMETRY_HINTS = re.compile(r"\btriangle\b|\bcircle\b|\bangle\b|\bpolygon\b|\bperpendicular\b", re.IGNORECASE)
NUMBER_THEORY_HINTS = re.compile(r"\bremainder\b|\bmod\b|\bdivisible\b|\bprime\b", re.IGNORECASE)


def default_policy_set() -> PolicySet:
    retry_reasons = (
        "weak_parse",
        "multiple_boxed",
        "suspicious_tiny_answer",
        "tool_mismatch",
        "inconsistent_reasoning",
    )
    retry_prompt_map = (
        ("weak_parse", "tool_guided"),
        ("multiple_boxed", "self_refute"),
        ("suspicious_tiny_answer", "self_refute"),
        ("tool_mismatch", "baseline"),
        ("inconsistent_reasoning", "self_refute"),
    )

    return PolicySet(
        easy=SolvePolicy(
            name="easy",
            attempt_policy=AttemptPolicy(
                attempt_count=3,
                prompt_family_distribution=("baseline", "tool_guided", "baseline"),
                tool_use_mode=TOOL_USE_OPTIONAL,
            ),
            early_stop_policy=EarlyStopPolicy(
                min_matching_answers=2,
                min_parse_tier="tier1",
                allowed_verify_statuses=("ok", "flagged"),
                require_tool_support=False,
            ),
            retry_policy=RetryPolicy(
                retry_budget=0,
                allowed_reasons=(),
                reason_prompt_family_map=(),
            ),
            selection_policy=SelectionPolicy(),
        ),
        medium=SolvePolicy(
            name="medium",
            attempt_policy=AttemptPolicy(
                attempt_count=5,
                prompt_family_distribution=(
                    "baseline",
                    "tool_guided",
                    "self_refute",
                    "tool_guided",
                    "baseline",
                ),
                tool_use_mode=TOOL_USE_ENCOURAGED,
            ),
            early_stop_policy=EarlyStopPolicy(
                min_matching_answers=3,
                min_parse_tier="tier1",
                allowed_verify_statuses=("ok", "flagged"),
                require_tool_support=False,
            ),
            retry_policy=RetryPolicy(
                retry_budget=1,
                allowed_reasons=retry_reasons,
                reason_prompt_family_map=retry_prompt_map,
            ),
            selection_policy=SelectionPolicy(),
        ),
        hard=SolvePolicy(
            name="hard",
            attempt_policy=AttemptPolicy(
                attempt_count=7,
                prompt_family_distribution=(
                    "baseline",
                    "tool_guided",
                    "self_refute",
                    "tool_guided",
                    "baseline",
                    "self_refute",
                    "tool_guided",
                ),
                tool_use_mode=TOOL_USE_REQUIRED,
            ),
            early_stop_policy=EarlyStopPolicy(
                min_matching_answers=4,
                min_parse_tier="tier1",
                allowed_verify_statuses=("ok",),
                require_tool_support=True,
            ),
            retry_policy=RetryPolicy(
                retry_budget=2,
                allowed_reasons=retry_reasons,
                reason_prompt_family_map=retry_prompt_map,
            ),
            selection_policy=SelectionPolicy(),
        ),
    )


def classify_problem_difficulty(problem_text: str) -> str:
    normalized = problem_text.strip()
    if len(normalized) >= 320 or GEOMETRY_HINTS.search(normalized):
        return "hard"
    if len(normalized) <= 120 and NUMBER_THEORY_HINTS.search(normalized):
        return "easy"
    return "medium"


def should_early_stop(attempts: list[AttemptRecord], policy: SolvePolicy) -> EarlyStopDecision:
    stop_policy = policy.early_stop_policy
    min_tier_rank = _parse_tier_rank(stop_policy.min_parse_tier)

    qualified_attempts = [
        attempt
        for attempt in attempts
        if attempt.parsed_answer is not None
        and attempt.parse_tier_rank >= min_tier_rank
        and attempt.verify_status in stop_policy.allowed_verify_statuses
    ]
    if not qualified_attempts:
        return EarlyStopDecision(False, None, (), None)

    grouped_attempts: dict[int, list[AttemptRecord]] = {}
    for attempt in qualified_attempts:
        grouped_attempts.setdefault(attempt.parsed_answer, []).append(attempt)

    best_answer = None
    best_group: list[AttemptRecord] = []
    for answer, group in grouped_attempts.items():
        if len(group) > len(best_group):
            best_answer = answer
            best_group = group

    if best_answer is None or len(best_group) < stop_policy.min_matching_answers:
        return EarlyStopDecision(False, None, (), None)

    if stop_policy.require_tool_support and not any(a.tool_consistency for a in best_group):
        return EarlyStopDecision(
            False,
            best_answer,
            tuple(a.attempt_index for a in best_group),
            None,
        )

    return EarlyStopDecision(
        True,
        best_answer,
        tuple(a.attempt_index for a in best_group),
        "verify_aware_consensus",
    )


def recommend_retry(
    attempts: list[AttemptRecord],
    policy: SolvePolicy,
    *,
    retries_used: int = 0,
) -> RetryDecision:
    retry_policy = policy.retry_policy
    remaining_budget = max(0, retry_policy.retry_budget - retries_used)
    if remaining_budget <= 0:
        return RetryDecision(False, None, None, remaining_budget)

    reason_counter: Counter[str] = Counter()
    for attempt in attempts:
        for reason in attempt.policy_retry_reasons:
            if reason in retry_policy.allowed_reasons:
                reason_counter[reason] += 1

    if not reason_counter:
        return RetryDecision(False, None, None, remaining_budget)

    prioritized_reasons = [
        reason for reason in retry_policy.allowed_reasons if reason in reason_counter
    ]
    chosen_reason = prioritized_reasons[0]
    prompt_family = _reason_prompt_family(retry_policy, chosen_reason)

    return RetryDecision(True, chosen_reason, prompt_family, remaining_budget)


def _reason_prompt_family(retry_policy: RetryPolicy, reason: str) -> str | None:
    for mapped_reason, prompt_family in retry_policy.reason_prompt_family_map:
        if mapped_reason == reason:
            return prompt_family
    return None


def _parse_tier_rank(parse_tier: str) -> int:
    ranks = {
        "none": 0,
        "tier3": 1,
        "tier2": 2,
        "tier1": 3,
    }
    return ranks.get(parse_tier, 0)

from dataclasses import dataclass



@dataclass(frozen=True, slots=True)
class RegressionAssessment:
    expected_answer: int
    selected_answer: int | None
    candidate_correct_exists: bool
    selected_is_correct: bool
    selection_failure: bool
    generation_failure: bool
    extraction_failure: bool


def assess_selection_outcome(
    attempts: list[AttemptRecord],
    selection_result: SelectionResult,
    *,
    expected_answer: int,
) -> RegressionAssessment:
    any_extraction_success = any(not attempt.extraction_failed for attempt in attempts)
    candidate_correct_exists = any(
        attempt.parsed_answer == expected_answer for attempt in attempts
    )
    selected_is_correct = selection_result.selected_answer == expected_answer
    selection_failure = candidate_correct_exists and not selected_is_correct
    generation_failure = not candidate_correct_exists
    extraction_failure = (
        generation_failure
        and not any_extraction_success
        and selection_result.selected_answer is None
    )

    return RegressionAssessment(
        expected_answer=expected_answer,
        selected_answer=selection_result.selected_answer,
        candidate_correct_exists=candidate_correct_exists,
        selected_is_correct=selected_is_correct,
        selection_failure=selection_failure,
        generation_failure=generation_failure,
        extraction_failure=extraction_failure,
    )

class AIMO3Solver:

    def __init__(self, cfg, port: int = 8000):
    
        self.cfg = cfg
        self.port = port
        self.base_url = f'http://0.0.0.0:{port}/v1'
        self.api_key = 'sk-local'
        self.template = AIMO3Template()
        self.encoding = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)
        self.stop_token_ids = self.encoding.stop_tokens_for_assistant_actions()
    
        self._preload_model_weights()
        
        self.server_process = self._start_server()
    
        self.client = OpenAI(
            base_url=self.base_url, 
            api_key=self.api_key, 
            timeout=self.cfg.session_timeout
        )
    
        self._wait_for_server()
        self._initialize_kernels()
    
        self.notebook_start_time = time.time()
        self.problems_remaining = 50
        self.policy_set = default_policy_set()
    
    def _preload_model_weights(self) -> None:
    
        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()
        
        files_to_load = []
        total_size = 0
    
        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)
    
                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)
    
        def _read_file(path: str) -> None:
    
            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))
    
        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')
    
    def _start_server(self) -> subprocess.Popen:
    
        cmd = [
            sys.executable, 
            '-m', 
            'vllm.entrypoints.openai.api_server', 
            '--seed', 
            str(self.cfg.seed), 
            '--model', 
            self.cfg.model_path, 
            '--served-model-name', 
            self.cfg.served_model_name, 
            '--tensor-parallel-size', 
            '1', 
            '--max-num-seqs', 
            str(self.cfg.batch_size), 
            '--gpu-memory-utilization', 
            str(self.cfg.gpu_memory_utilization), 
            '--host', 
            '0.0.0.0', 
            '--port', 
            str(self.port), 
            '--dtype', 
            self.cfg.dtype, 
            '--kv-cache-dtype', 
            self.cfg.kv_cache_dtype, 
            '--max-model-len', 
            str(self.cfg.context_tokens), 
            '--stream-interval', 
            str(self.cfg.stream_interval), 
            '--async-scheduling', 
            '--disable-log-stats', 
            '--enable-prefix-caching'
        ]
    
        self.log_file = open('vllm_server.log', 'w')
    
        return subprocess.Popen(
            cmd, 
            stdout=self.log_file, 
            stderr=subprocess.STDOUT, 
            start_new_session=True
        )
    
    def _wait_for_server(self):
    
        print('Waiting for vLLM server...')
        start_time = time.time()
    
        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()
    
            if return_code is not None:
                self.log_file.flush()
    
                with open('vllm_server.log', 'r') as log_file:
                    logs = log_file.read()
    
                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')
    
            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
    
                return
    
            except Exception:
                time.sleep(1)
    
        raise RuntimeError('Server failed to start (timeout).\n')
    
    def _initialize_kernels(self) -> None:
    
        print(f'Initializing {self.cfg.workers} persistent Jupyter kernels...')
        start_time = time.time()
    
        self.sandbox_pool = queue.Queue()
    
        def _create_sandbox():
            
            return AIMO3Sandbox(timeout=self.cfg.jupyter_timeout)
    
        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            futures = [executor.submit(_create_sandbox) for _ in range(self.cfg.workers)]
    
            for future in as_completed(futures):
                self.sandbox_pool.put(future.result())
    
        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds.\n')
    
    def _scan_for_answer_legacy(self, text: str) -> int | None:
        
        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
                
        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
    
        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)
    
                if 0 <= value <= 99999:
                    return value
    
            except ValueError:
                pass
    
        return None

    def _scan_for_answer(self, problem: str, text: str) -> int | None:
        final_answer, parser_result, modulus_result, verify_result = postprocess_candidate(problem, text)

        if final_answer is not None:
            return final_answer

        return self._scan_for_answer_legacy(text)
    
    def _compute_mean_entropy(self, logprobs_buffer: list) -> float:
    
        if not logprobs_buffer:
            return float('inf')
    
        total_entropy = 0.0
        token_count = 0
    
        for top_logprobs_dict in logprobs_buffer:
            
            if not isinstance(top_logprobs_dict, dict):
                continue
            
            if not top_logprobs_dict:
                continue
            
            token_entropy = 0.0
            
            for token_str, log_prob in top_logprobs_dict.items():
                prob = math.exp(log_prob)
                
                if prob > 0:
                    token_entropy -= prob * math.log2(prob)
            
            total_entropy += token_entropy
            token_count += 1
    
        if token_count == 0:
            return float('inf')
    
        return total_entropy / token_count
    
    def _process_attempt(
        self, 
        raw_problem_text: str, 
        prompted_problem_text: str, 
        system_prompt: str, 
        prompt_family: str, 
        attempt_index: int, 
        stop_event: threading.Event, 
        deadline: float,
        metadata: Optional[dict] = None
    ) -> dict:
    
        if stop_event.is_set() or time.time() > deadline:
            attempt_record = build_attempt_record(
                attempt_index=attempt_index + 1,
                prompt_family=prompt_family,
                raw_problem_text=raw_problem_text,
                prompted_problem_text=prompted_problem_text,
                raw_output_text='',
                response_length=0,
                entropy=float('inf'),
                finish_reason='cancelled',
                metadata=metadata,
            )
            return {
                'Attempt': attempt_index + 1, 
                'Prompt Family': prompt_family,
                'Answer': None, 
                'Python Calls': 0, 
                'Python Errors': 0, 
                'Response Length': 0, 
                'Entropy': float('inf'),
                'Parse Tier': attempt_record.parse_tier,
                'Verify Status': attempt_record.verify_status,
                'Retry Reasons': ', '.join(attempt_record.policy_retry_reasons),
                'AttemptRecord': attempt_record
            }
    
        local_tool = None
        sandbox = None
        python_calls = 0
        python_errors = 0
        total_tokens = 0
        final_answer = None
        full_output_chunks = []
        
        logprobs_buffer = []
    
        attempt_seed = int(math.pow(self.cfg.seed + attempt_index, 2))
    
        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg.sandbox_timeout)
    
            local_tool = AIMO3Tool(
                local_jupyter_timeout=self.cfg.jupyter_timeout, 
                tool_prompt=self.cfg.tool_prompt, 
                sandbox=sandbox
            )
    
            encoding = self.encoding
            messages = self.template.apply_chat_template(
                system_prompt, 
                prompted_problem_text, 
                local_tool.tool_config
            )
    
            conversation = Conversation.from_messages(messages)
    
            for _ in range(self.cfg.turns):
                if stop_event.is_set() or time.time() > deadline:
                    break
    
                prompt_ids = encoding.render_conversation_for_completion(conversation, Role.ASSISTANT)
                max_tokens = self.cfg.context_tokens - len(prompt_ids)
    
                if max_tokens < self.cfg.buffer_tokens:
                    break
    
                stream = self.client.completions.create(
                    model=self.cfg.served_model_name, 
                    temperature=self.cfg.temperature, 
                    logprobs=self.cfg.top_logprobs, 
                    max_tokens=max_tokens, 
                    prompt=prompt_ids, 
                    seed=attempt_seed, 
                    stream=True, 
                    extra_body={
                        'min_p': self.cfg.min_p, 
                        'stop_token_ids': self.stop_token_ids, 
                        'return_token_ids': True
                    }
                )
    
                try:
                    token_buffer = []
                    text_chunks = []
    
                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
    
                        new_tokens = chunk.choices[0].token_ids
                        new_text = chunk.choices[0].text
    
                        if new_tokens:
                            token_buffer.extend(new_tokens)
                            total_tokens += len(new_tokens)
                            text_chunks.append(new_text)
                            full_output_chunks.append(new_text)
                            
                            chunk_logprobs = chunk.choices[0].logprobs
                            
                            if chunk_logprobs is not None:
                                if chunk_logprobs.top_logprobs:
                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)
    
                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg.search_tokens:])
                            answer = self._scan_for_answer(raw_problem_text, search_text)
    
                            if answer is not None:
                                final_answer = answer
                                break
    
                finally:
                    stream.close()
    
                if final_answer is not None:
                    break
    
                if not token_buffer:
                    break
    
                new_messages = encoding.parse_messages_from_completion_tokens(token_buffer, Role.ASSISTANT)
                conversation.messages.extend(new_messages)
                last_message = new_messages[-1]
    
                if last_message.channel == 'final':
                    answer_text = last_message.content[0].text
                    full_output_chunks.append(answer_text)
                    final_answer = self._scan_for_answer(raw_problem_text, answer_text)
                    break
    
                if last_message.recipient == 'python':
                    python_calls += 1
                    tool_responses = local_tool.process_sync_plus(last_message)
    
                    response_text = tool_responses[0].content[0].text
    
                    if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                        python_errors += 1
    
                    conversation.messages.extend(tool_responses)
    
        except Exception as exc:
            python_errors += 1
    
        finally:
            if sandbox is not None:
                sandbox.reset()
                self.sandbox_pool.put(sandbox)
    
        mean_entropy = self._compute_mean_entropy(logprobs_buffer)
        raw_output_text = ''.join(full_output_chunks)

        if not raw_output_text and final_answer is not None:
            raw_output_text = f'Final answer is \\boxed{{{final_answer}}}.'

        attempt_record = build_attempt_record(
            attempt_index=attempt_index + 1,
            prompt_family=prompt_family,
            raw_problem_text=raw_problem_text,
            prompted_problem_text=prompted_problem_text,
            raw_output_text=raw_output_text,
            tool_used=python_calls > 0,
            python_calls=python_calls,
            python_errors=python_errors,
            response_length=total_tokens,
            entropy=mean_entropy,
            finish_reason='parsed_answer_found' if final_answer is not None else 'stream_end',
        )
    
        return {
            'Attempt': attempt_record.attempt_index, 
            'Prompt Family': attempt_record.prompt_family,
            'Response Length': total_tokens, 
            'Python Calls': python_calls, 
            'Python Errors': python_errors, 
            'Entropy': mean_entropy, 
            'Answer': attempt_record.parsed_answer,
            'Parse Tier': attempt_record.parse_tier,
            'Verify Status': attempt_record.verify_status,
            'Retry Reasons': ', '.join(attempt_record.policy_retry_reasons),
            'AttemptRecord': attempt_record
        }
    
    def _select_answer(self, attempt_records: list) -> int:

        selection_result = select_best_attempt(attempt_records)
        vote_rows = []

        for breakdown in selection_result.scored_attempts:
            record = next(
                attempt_record
                for attempt_record in attempt_records
                if attempt_record.attempt_index == breakdown.attempt_index
            )
            vote_rows.append({
                'Attempt': breakdown.attempt_index,
                'Prompt Family': record.prompt_family,
                'Answer': breakdown.answer,
                'Score': round(breakdown.total_score, 3),
                'Parse Tier': record.parse_tier,
                'Verify Status': record.verify_status,
            })

        vote_dataframe = pd.DataFrame(vote_rows)
        if not vote_dataframe.empty:
            display(vote_dataframe)

        final_answer = selection_result.selected_answer if selection_result.selected_answer is not None else 0
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer
    
    def solve_problem(self, problem: str, return_trace: bool = False):
    
        print(f'\nProblem: {problem}\n')
        
        raw_problem_text = problem
        prompted_problem_text = f'{raw_problem_text} {self.cfg.preference_prompt}'
        policy = self.policy_set.for_problem(raw_problem_text)
        prompt_routes = build_prompt_routes_for_policy(policy)
    
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg.base_problem_timeout
    
        budget = time_left - reserved_time
        budget = min(budget, self.cfg.high_problem_timeout)
        budget = max(budget, self.cfg.base_problem_timeout)
    
        deadline = time.time() + budget
    
        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')
    
        tasks = []
    
        for attempt_index, prompt_route in enumerate(prompt_routes):
            tasks.append((prompt_route, attempt_index))
    
        detailed_results = []
        attempt_records = []
        retries_used = 0
        retry_history = set()
    
        stop_event = threading.Event()
    
        executor = ThreadPoolExecutor(max_workers=self.cfg.workers)
    
        try:
            futures = []
    
            for (prompt_route, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt, 
                    raw_problem_text,
                    prompted_problem_text,
                    prompt_route.full_prompt, 
                    prompt_route.family_name,
                    attempt_index, 
                    stop_event, 
                    deadline,
                    metadata={'phase': 'initial'}
                )
    
                futures.append(future)
    
            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)
                    attempt_records.append(result['AttemptRecord'])

                    early_stop_decision = should_early_stop(attempt_records, policy)

                    if early_stop_decision.should_stop:
                        stop_event.set()
    
                        for f in futures:
                            f.cancel()
    
                        break
    
                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue
    
        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)
            
            self.problems_remaining = max(0, self.problems_remaining - 1)

        while time.time() < deadline:
            retry_decision = recommend_retry(attempt_records, policy, retries_used=retries_used)
            if not retry_decision.should_retry:
                break

            retry_key = (retry_decision.reason, retry_decision.prompt_family)
            if retry_key in retry_history:
                break

            retry_route = build_retry_prompt_route(policy, retry_decision)
            if retry_route is None:
                break

            retry_history.add(retry_key)
            retry_result = self._process_attempt(
                raw_problem_text,
                prompted_problem_text,
                retry_route.full_prompt,
                retry_route.family_name,
                len(attempt_records),
                threading.Event(),
                deadline,
                metadata={'phase': 'retry', 'retry_reason': retry_decision.reason},
            )
            detailed_results.append(retry_result)
            attempt_records.append(retry_result['AttemptRecord'])
            retries_used += 1

            early_stop_decision = should_early_stop(attempt_records, policy)
            if early_stop_decision.should_stop:
                break
    
        if detailed_results:
            display_rows = []
            for result in detailed_results:
                display_rows.append({
                    key: value
                    for key, value in result.items()
                    if key != 'AttemptRecord'
                })
            results_dataframe = pd.DataFrame(display_rows)
            results_dataframe['Entropy'] = results_dataframe['Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')
            
            display(results_dataframe)
    
        if not attempt_records or not any(record.parsed_answer is not None for record in attempt_records):
            print('\nResult: 0\n')
            selection_result = select_best_attempt(attempt_records)
            if return_trace:
                return {
                    'selected_answer': 0,
                    'selection_result': selection_result,
                    'attempt_records': attempt_records,
                    'detailed_results': detailed_results,
                    'policy': policy,
                    'retries_used': retries_used,
                }
            return 0

        if return_trace:
            selection_result = select_best_attempt(attempt_records)
            final_answer = selection_result.selected_answer if selection_result.selected_answer is not None else 0
            return {
                'selected_answer': final_answer,
                'selection_result': selection_result,
                'attempt_records': attempt_records,
                'detailed_results': detailed_results,
                'policy': policy,
                'retries_used': retries_used,
            }

        return self._select_answer(attempt_records)
    
    def __del__(self):
    
        if hasattr(self, 'server_process'):
            self.server_process.terminate()
            self.server_process.wait()
    
        if hasattr(self, 'log_file'):
            self.log_file.close()
    
        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()
    
                except Exception:
                    pass


In [15]:
solver = AIMO3Solver(CFG)

Loading model weights from /kaggle/input/gpt-oss-120b/transformers/default/1 into OS Page Cache...
Processed 26 files (65.28 GB) in 99.67 seconds.

Waiting for vLLM server...
Server is ready (took 126.11 seconds).

Initializing 16 persistent Jupyter kernels...
Kernels initialized in 2.95 seconds.



# Reference Evaluation

This notebook is evaluation-only. It runs the current solver on the 10 reference problems, writes attempt traces, and summarizes failure modes.


In [ ]:
import csv
import json
from io import StringIO
from pathlib import Path

REFERENCE_CSV_FALLBACK = '"id","problem","answer"\n"0e644e","Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie on segments $BC$ and $AC$, respectively, such that $AD=AE=AB$. Line $DE$ intersects $AB$ at $X$. Circles $BXD$ and $CED$ intersect for the second time at $Y \\neq D$. Suppose that $Y$ lies on line $AD$. There is a unique such triangle with minimal perimeter. This triangle has side lengths $a=BC$, $b=CA$, and $c=AB$. Find the remainder when $abc$ is divided by $10^{5}$.",336\n"26de63","Define a function $f \\colon \\mathbb{Z}_{\\geq 1} \\to \\mathbb{Z}_{\\geq 1}$ by\n\\begin{equation*}\n    f(n) = \\sum_{i = 1}^n \\sum_{j = 1}^n j^{1024} \\left\\lfloor\\frac1j + \\frac{n-i}{n}\\right\\rfloor.\n\\end{equation*}\nLet $M=2 \\cdot 3 \\cdot 5 \\cdot 7 \\cdot 11 \\cdot 13$ and let $N = f{\\left(M^{15}\\right)} - f{\\left(M^{15}-1\\right)}$. Let $k$ be the largest non-negative integer such that $2^k$ divides $N$. What is the remainder when $2^k$ is divided by $5^7$?",32951\n"424e18","A tournament is held with $2^{20}$ runners each of which has a different running speed. In each race, two runners compete against each other with the faster runner always winning the race. The competition consists of $20$ rounds with each runner starting with a score of $0$. In each round, the runners are paired in such a way that in each pair, both runners have the same score at the beginning of the round. The winner of each race in the $i^{\\text{th}}$ round receives $2^{20-i}$ points and the loser gets no points.\n\nAt the end of the tournament, we rank the competitors according to their scores. Let $N$ denote the number of possible orderings of the competitors at the end of the tournament. Let $k$ be the largest positive integer such that $10^k$ divides $N$. What is the remainder when $k$ is divided by $10^{5}$?",21818\n"42d360","On a blackboard, Ken starts off by writing a positive integer $n$ and then applies the following move until he first reaches $1$. Given that the number on the board is $m$, he chooses a base $b$, where $2 \\leq b \\leq m$, and considers the unique base-$b$ representation of $m$,\n\\begin{equation*}\n    m = \\sum_{k = 0}^\\infty a_k \\cdot b^k\n\\end{equation*}\nwhere $a_k$ are non-negative integers and $0 \\leq a_k < b$ for each $k$. Ken then erases $m$ on the blackboard and replaces it with $\\sum\\limits_{k = 0}^\\infty a_k$.\n\nAcross all choices of $1 \\leq n \\leq 10^{10^5}$, the largest possible number of moves Ken could make is $M$. What is the remainder when $M$ is divided by $10^{5}$?",32193\n"641659","Let $ABC$ be a triangle with $AB \\neq AC$, circumcircle $\\Omega$, and incircle $\\omega$. Let the contact points of $\\omega$ with $BC$, $CA$, and $AB$ be $D$, $E$, and $F$, respectively. Let the circumcircle of $AFE$ meet $\\Omega$ at $K$ and let the reflection of $K$ in $EF$ be $K\'$. Let $N$ denote the foot of the perpendicular from $D$ to $EF$. The circle tangent to line $BN$ and passing through $B$ and $K$ intersects $BC$ again at $T \\neq B$. \n    \nLet sequence $(F_n)_{n \\geq 0}$ be defined by $F_0 = 0$, $F_1 = 1$ and for $n \\geq 2$, $F_n = F_{n-1} + F_{n-2}$. Call $ABC$ $n$\\emph{-tastic} if $BD = F_n$, $CD = F_{n+1}$, and $KNK\'B$ is cyclic. Across all $n$-tastic triangles, let $a_n$ denote the maximum possible value of $\\frac{CT \\cdot NB}{BT \\cdot NE}$. Let $\\alpha$ denote the smallest real number such that for all sufficiently large $n$, $a_{2n} < \\alpha$. Given that $\\alpha = p + \\sqrt{q}$ for rationals $p$ and $q$, what is the remainder when $\\left\\lfloor p^{q^p} \\right\\rfloor$ is divided by $99991$?",57447\n"86e8e5","Let $n \\geq 6$ be a positive integer. We call a positive integer $n$-Norwegian if it has three distinct positive divisors whose sum is equal to $n$. Let $f(n)$ denote the smallest $n$-Norwegian positive integer. Let $M=3^{2025!}$ and for a non-negative integer $c$ define \n\\begin{equation*}\n    g(c)=\\frac{1}{2025!}\\left\\lfloor \\frac{2025! f(M+c)}{M}\\right\\rfloor.\n\\end{equation*}\nWe can write \n\\begin{equation*}\n    g(0)+g(4M)+g(1848374)+g(10162574)+g(265710644)+g(44636594)=\\frac{p}{q}\n\\end{equation*}\nwhere $p$ and $q$ are coprime positive integers. What is the remainder when $p+q$ is divided by $99991$?",8687\n"92ba6a","Alice and Bob are each holding some integer number of sweets. Alice says to Bob: ``If we each added the number of sweets we\'re holding to our (positive integer) age, my answer would be double yours. If we took the product, then my answer would be four times yours.\'\' Bob replies: ``Why don\'t you give me five of your sweets because then both our sum and product would be equal.\'\' What is the product of Alice and Bob\'s ages?",50\n"9c1c5f","Let $f \\colon \\mathbb{Z}_{\\geq 1} \\to \\mathbb{Z}_{\\geq 1}$ be a function such that for all positive integers $m$ and $n$, \n\\begin{equation*}\n    f(m) + f(n) = f(m + n + mn).\n\\end{equation*}\nAcross all functions $f$ such that $f(n) \\leq 1000$ for all $n \\leq 1000$, how many different values can $f(2024)$ take?",580\n"a295e9","A $500 \\times 500$ square is divided into $k$ rectangles, each having integer side lengths. Given that no two of these rectangles have the same perimeter, the largest possible value of $k$ is $\\mathcal{K}$. What is the remainder when $k$ is divided by $10^{5}$?",520\n"dd7f5e","Let $\\mathcal{F}$ be the set of functions $\\alpha \\colon \\mathbb{Z}\\to \\mathbb{Z}$ for which there are only finitely many $n \\in \\mathbb{Z}$ such that $\\alpha(n) \\neq 0$. \n\nFor two functions $\\alpha$ and $\\beta$ in $\\mathcal{F}$, define their product $\\alpha\\star\\beta$ to be $\\sum\\limits_{n\\in\\mathbb{Z}} \\alpha(n)\\cdot \\beta(n)$. Also, for $n\\in\\mathbb{Z}$, define a shift operator $S_n \\colon \\mathcal{F}\\to \\mathcal{F}$ by $S_n(\\alpha)(t)=\\alpha(t+n)$ for all $t \\in \\mathbb{Z}$.\n\nA function $\\alpha \\in \\mathcal{F}$ is called \\emph{shifty} if \n\\begin{itemize}\n    \\item $\\alpha(m)=0$ for all integers $m<0$ and $m>8$ and\n    \\item There exists $\\beta \\in \\mathcal{F}$ and integers $k \\neq l$ such that for all $n \\in \\mathbb{Z}$\n    \\begin{equation*}\n        S_n(\\alpha)\\star\\beta =\n        \\begin{cases}\n            1 & n \\in \\{k,l\\} \\\\\n            0 & n \\not \\in \\{k,l\\}\n        \\end{cases}\n        \\; .\n    \\end{equation*}\n\\end{itemize}\nHow many shifty functions are there in $\\mathcal{F}$?",160'


def write_jsonl(path: str | Path, records):
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(dict(record), ensure_ascii=True, sort_keys=True))
            handle.write("\n")
    return output_path


def write_csv_report(path: str | Path, rows, *, fieldnames: list[str]):
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)
    return output_path


def write_json(path: str | Path, payload):
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as handle:
        json.dump(dict(payload), handle, ensure_ascii=True, indent=2, sort_keys=True)
        handle.write("\n")
    return output_path


def resolve_artifacts_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'data/reference/reference.csv').exists():
            return candidate
    return cwd


def load_reference_df() -> pd.DataFrame:
    cwd = Path.cwd().resolve()
    for candidate in [cwd / 'data/reference/reference.csv', *[(parent / 'data/reference/reference.csv') for parent in [cwd, *cwd.parents]]]:
        if candidate.exists():
            print(f'Using reference CSV from: {candidate}')
            return pd.read_csv(candidate)
    print('Local reference.csv not found; using embedded fallback copy.')
    return pd.read_csv(StringIO(REFERENCE_CSV_FALLBACK))


ARTIFACTS_ROOT = resolve_artifacts_root()
ARTIFACTS_DIR = ARTIFACTS_ROOT / 'artifacts' / 'reference_eval_current_pipeline'
ATTEMPT_TRACES_PATH = ARTIFACTS_DIR / 'attempt_traces.jsonl'
RUN_SUMMARY_CSV_PATH = ARTIFACTS_DIR / 'run_summary.csv'
EVAL_SUMMARY_JSON_PATH = ARTIFACTS_DIR / 'evaluation_summary.json'

RUN_SUMMARY_FIELDS = [
    'problem_id',
    'gold_answer',
    'selected_answer',
    'is_correct',
    'candidate_correct_exists',
    'selection_failure',
    'generation_failure',
    'extraction_failure',
    'retry_recoverable',
    'num_attempts',
    'retries_used',
    'selected_prompt_family',
    'selected_verify_status',
    'selected_parse_tier',
]


def serialize_attempt_trace(problem_id: str, gold_answer: int, selected_answer: int, attempt):
    return {
        'problem_id': problem_id,
        'gold_answer': gold_answer,
        'selected_answer': selected_answer,
        'attempt_index': attempt.attempt_index,
        'prompt_family': attempt.prompt_family,
        'parsed_answer': attempt.parsed_answer,
        'parse_tier': attempt.parse_tier,
        'verify_status': attempt.verify_status,
        'retry_reasons': list(attempt.policy_retry_reasons),
        'suspicious_flags': list(attempt.suspicious_flags),
        'entropy': None if attempt.entropy == float('inf') else attempt.entropy,
        'tool_used': attempt.tool_used,
        'tool_consistency': attempt.tool_consistency,
        'python_calls': attempt.python_calls,
        'python_errors': attempt.python_errors,
        'finish_reason': attempt.finish_reason,
        'is_retry': attempt.metadata.get('phase') == 'retry',
        'retry_reason': attempt.metadata.get('retry_reason'),
    }


def detect_retry_recoverable(attempt_records, expected_answer: int) -> bool:
    initial_correct_exists = any(
        attempt.parsed_answer == expected_answer and attempt.metadata.get('phase') != 'retry'
        for attempt in attempt_records
    )
    retry_correct_exists = any(
        attempt.parsed_answer == expected_answer and attempt.metadata.get('phase') == 'retry'
        for attempt in attempt_records
    )
    return (not initial_correct_exists) and retry_correct_exists


def evaluate_reference_set(solver):
    reference_df = load_reference_df()
    attempt_trace_rows = []
    run_summary_rows = []

    aggregate = {
        'num_questions': 0,
        'num_correct': 0,
        'candidate_correct_exists': 0,
        'selection_failure': 0,
        'generation_failure': 0,
        'extraction_failure': 0,
        'retry_recoverable': 0,
    }

    for row in reference_df.itertuples(index=False):
        problem_id = row.id
        problem_text = row.problem
        gold_answer = int(row.answer)

        solve_result = solver.solve_problem(problem_text, return_trace=True)
        attempt_records = solve_result['attempt_records']
        selection_result = solve_result['selection_result']
        selected_answer = int(solve_result['selected_answer'])

        assessment = assess_selection_outcome(
            attempt_records,
            selection_result,
            expected_answer=gold_answer,
        )
        retry_recoverable = detect_retry_recoverable(attempt_records, gold_answer)

        selected_attempt = selection_result.selected_attempt
        run_summary_rows.append({
            'problem_id': problem_id,
            'gold_answer': gold_answer,
            'selected_answer': selected_answer,
            'is_correct': selected_answer == gold_answer,
            'candidate_correct_exists': assessment.candidate_correct_exists,
            'selection_failure': assessment.selection_failure,
            'generation_failure': assessment.generation_failure,
            'extraction_failure': assessment.extraction_failure,
            'retry_recoverable': retry_recoverable,
            'num_attempts': len(attempt_records),
            'retries_used': solve_result['retries_used'],
            'selected_prompt_family': selected_attempt.prompt_family if selected_attempt else None,
            'selected_verify_status': selected_attempt.verify_status if selected_attempt else None,
            'selected_parse_tier': selected_attempt.parse_tier if selected_attempt else None,
        })

        for attempt in attempt_records:
            attempt_trace_rows.append(serialize_attempt_trace(problem_id, gold_answer, selected_answer, attempt))

        aggregate['num_questions'] += 1
        aggregate['num_correct'] += int(selected_answer == gold_answer)
        aggregate['candidate_correct_exists'] += int(assessment.candidate_correct_exists)
        aggregate['selection_failure'] += int(assessment.selection_failure)
        aggregate['generation_failure'] += int(assessment.generation_failure)
        aggregate['extraction_failure'] += int(assessment.extraction_failure)
        aggregate['retry_recoverable'] += int(retry_recoverable)

    aggregate['accuracy'] = (aggregate['num_correct'] / aggregate['num_questions']) if aggregate['num_questions'] else 0.0
    aggregate['attempt_traces_path'] = str(ATTEMPT_TRACES_PATH)
    aggregate['run_summary_csv_path'] = str(RUN_SUMMARY_CSV_PATH)
    aggregate['eval_summary_json_path'] = str(EVAL_SUMMARY_JSON_PATH)

    write_jsonl(ATTEMPT_TRACES_PATH, attempt_trace_rows)
    write_csv_report(RUN_SUMMARY_CSV_PATH, run_summary_rows, fieldnames=RUN_SUMMARY_FIELDS)
    write_json(EVAL_SUMMARY_JSON_PATH, aggregate)

    summary_df = pd.DataFrame(run_summary_rows)
    aggregate_df = pd.DataFrame([aggregate])
    display(summary_df)
    display(aggregate_df)

    print(f'Attempt traces written to: {ATTEMPT_TRACES_PATH}')
    print(f'Run summary written to: {RUN_SUMMARY_CSV_PATH}')
    print(f'Aggregate summary written to: {EVAL_SUMMARY_JSON_PATH}')

    return {
        'run_summary_rows': run_summary_rows,
        'attempt_trace_rows': attempt_trace_rows,
        'aggregate_summary': aggregate,
    }


In [ ]:
reference_eval_results = evaluate_reference_set(solver)
reference_eval_results['aggregate_summary']
